# Modeling — Prévision des Ventes CA
## Preprocessing → Augmentation → Training → Prédiction

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import clickhouse_connect
import warnings
import json
import joblib

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.utils import shuffle

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:,.2f}'.format)
plt.rcParams['figure.figsize'] = (14, 5)
print('✅ Imports OK')

## 1. Chargement depuis ClickHouse

In [ ]:
client = clickhouse_connect.get_client(
    host='clickhouse', port=8123,
    user='default', password='clickhouse123',
    database='gold'
)

df = client.query_df("""
SELECT
    d.Year                               AS year,
    d.Month                              AS month,
    d.Quarter                            AS quarter,
    SUM(f.LinePrice)                     AS total_sales,
    SUM(f.Quantity)                      AS total_quantity,
    COUNT(DISTINCT f.DocumentCode)       AS nb_documents,
    COUNT(DISTINCT f.ClientSK)           AS nb_clients
FROM gold.fact_sales AS f
JOIN gold.dim_date AS d ON f.DateSK = d.DateSK
WHERE f.DocumentTypeSK IN (4, 1, 5)
  AND f.LinePrice IS NOT NULL
GROUP BY year, month, quarter
ORDER BY year, month
""")

df['date'] = pd.to_datetime(df[['year', 'month']].assign(day=1))
print(f'✅ {len(df)} mois chargés')
df.head()

## 2. Nettoyage — Jan 2022 → Déc 2025

In [ ]:
print(f'Avant nettoyage : {len(df)} mois')
print('Mois incomplets :')
print(df[
    (df['date'] < '2022-01-01') | (df['date'] >= '2026-01-01')
][['date', 'total_sales']].to_string(index=False))

df = df[
    (df['date'] >= '2022-01-01') & (df['date'] < '2026-01-01')
].reset_index(drop=True)

print(f'\nAprès nettoyage : {len(df)} mois')
print(f'Période : {df["date"].min().strftime("%b %Y")} → {df["date"].max().strftime("%b %Y")}')

## 3. Feature Engineering

In [ ]:
df_ml = df[[
    'date', 'year', 'month', 'quarter',
    'total_sales',
    'total_quantity', 'nb_documents', 'nb_clients',
]].copy()

# Variables temporelles
df_ml['semester']    = df_ml['month'].apply(lambda x: 1 if x <= 6 else 2)
df_ml['is_summer']   = df_ml['month'].isin([7, 8]).astype(int)
df_ml['is_end_year'] = df_ml['month'].isin([11, 12]).astype(int)
df_ml['is_january']  = (df_ml['month'] == 1).astype(int)

# Lag features
df_ml['lag_1']  = df_ml['total_sales'].shift(1)
df_ml['lag_12'] = df_ml['total_sales'].shift(12)

# lag_12 fallback sur moyenne mensuelle
monthly_avg = df_ml.groupby('month')['total_sales'].mean()
df_ml['lag_12'] = df_ml.apply(
    lambda row: monthly_avg[row['month']] if pd.isna(row['lag_12']) else row['lag_12'],
    axis=1
)

# Rolling features
df_ml['rolling_mean_3'] = df_ml['total_sales'].shift(1).rolling(3).mean()

df_ml = df_ml.dropna().reset_index(drop=True)

print(f'✅ Dataset prêt : {df_ml.shape}')
print(f'Période : {df_ml["date"].min().strftime("%b %Y")} → {df_ml["date"].max().strftime("%b %Y")}')
df_ml.head()

## 4. Corrélation des variables avec total_sales

In [ ]:
TARGET = 'total_sales'

FEATURES = [
    'month', 'quarter', 'semester',
    'total_quantity', 'nb_documents', 'nb_clients',
    'is_summer', 'is_end_year', 'is_january',
    'lag_1', 'lag_12', 'rolling_mean_3'
]

corr = df_ml[FEATURES + [TARGET]].corr()[TARGET].drop(TARGET).sort_values()

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['red' if v < 0 else 'steelblue' for v in corr.values]
ax.barh(corr.index, corr.values, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('📊 Corrélation des variables avec le CA')
ax.set_xlabel('Corrélation')
plt.tight_layout()
plt.show()

print('\nCorrélations :')
print(corr.to_string())

## 5. Sélection des features — toutes gardées

In [ ]:
# Garder toutes les features — pas de filtrage par seuil
FEATURES_FINAL = FEATURES

print(f'Variables sélectionnées ({len(FEATURES_FINAL)}) :')
for f in FEATURES_FINAL:
    print(f'  • {f:<20} corr = {corr[f]:.3f}')

# Variables contemporaines dans les features
CONTEMPORAINES = [c for c in ['nb_clients', 'nb_documents', 'total_quantity']
                  if c in FEATURES_FINAL]
print(f'\nContemporaines : {CONTEMPORAINES}')

print(f'\nX shape : {df_ml[FEATURES_FINAL].shape}')
print(f'y shape : {df_ml[TARGET].shape}')

## 6. Augmentation des données — Jittering ±1.5%

In [ ]:
def augment_dataset(df_orig, n_copies=3, noise_pct=0.015):
    """
    Crée n_copies copies avec bruit gaussien ±1.5% sur total_sales
    Recalcule les lags et rolling sur les nouvelles valeurs
    """
    np.random.seed(42)
    augmented = [df_orig.copy()]

    for _ in range(n_copies):
        df_copy = df_orig.copy()

        # Bruit sur total_sales
        noise = np.random.normal(0, noise_pct, len(df_copy))
        df_copy['total_sales'] = df_copy['total_sales'] * (1 + noise)

        # Recalculer les lags
        df_copy['lag_1']  = df_copy['total_sales'].shift(1)
        df_copy['lag_12'] = df_copy['total_sales'].shift(12)

        # lag_12 fallback
        monthly_avg_copy = df_copy.groupby('month')['total_sales'].mean()
        df_copy['lag_12'] = df_copy.apply(
            lambda row: monthly_avg_copy[row['month']]
            if pd.isna(row['lag_12']) else row['lag_12'], axis=1
        )

        # Rolling
        df_copy['rolling_mean_3'] = df_copy['total_sales'].shift(1).rolling(3).mean()
        df_copy = df_copy.dropna().reset_index(drop=True)

        augmented.append(df_copy)

    df_aug = pd.concat(augmented, ignore_index=True)

    print(f'Dataset original  : {len(df_orig)} mois')
    print(f'Copies créées     : {n_copies}')
    print(f'Dataset augmenté  : {len(df_aug)} lignes')
    print(f'CA moyen original : {df_orig["total_sales"].mean():,.2f} DT')
    print(f'CA moyen augmenté : {df_aug["total_sales"].mean():,.2f} DT')
    return df_aug

df_ml_aug = augment_dataset(df_ml, n_copies=3, noise_pct=0.015)

# Visualiser l'effet du bruit
n_orig = len(df_ml)
fig, ax = plt.subplots()
ax.plot(df_ml['date'], df_ml['total_sales'],
        color='steelblue', linewidth=2.5, label='Original', zorder=5)
for i in range(1, 4):
    subset = df_ml_aug.iloc[n_orig*i : n_orig*(i+1)]
    if len(subset) == n_orig:
        ax.plot(df_ml['date'], subset['total_sales'].values,
                color='orange', alpha=0.4, linewidth=1,
                label='Copie augmentée' if i == 1 else '')
ax.set_title('📊 Original vs Copies augmentées (±1.5% bruit gaussien)')
ax.set_ylabel('CA (DT)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.legend()
plt.tight_layout()
plt.show()

## 7. Sauvegarde datasets

In [ ]:
# Original → pour la prédiction future
df_ml.to_csv('/app/notebooks/ml_dataset.csv', index=False)
# Augmenté → pour le training
df_ml_aug.to_csv('/app/notebooks/ml_dataset_aug.csv', index=False)

with open('/app/notebooks/selected_features.json', 'w') as f:
    json.dump(list(FEATURES_FINAL), f)

print(f'✅ Dataset original sauvegardé  : {df_ml.shape}')
print(f'✅ Dataset augmenté sauvegardé  : {df_ml_aug.shape}')
print(f'✅ Features sauvegardées        : {FEATURES_FINAL}')

## 8. Chargement pour training

In [ ]:
# Dataset augmenté pour train
df_ml_aug = pd.read_csv('/app/notebooks/ml_dataset_aug.csv', parse_dates=['date'])
# Dataset original pour test et prédiction
df_ml     = pd.read_csv('/app/notebooks/ml_dataset.csv',     parse_dates=['date'])

with open('/app/notebooks/selected_features.json') as f:
    FEATURES = json.load(f)

TARGET         = 'total_sales'
CONTEMPORAINES = [c for c in ['nb_clients', 'nb_documents', 'total_quantity']
                  if c in FEATURES]

print(f'✅ Dataset augmenté : {df_ml_aug.shape}')
print(f'✅ Dataset original : {df_ml.shape}')
print(f'✅ Features ({len(FEATURES)}) : {FEATURES}')
print(f'✅ Contemporaines  : {CONTEMPORAINES}')

## 9. Split temporel — Train augmenté / Test original

In [ ]:
n_original = len(df_ml)
split_idx  = int(n_original * 0.9)
split_date = df_ml.iloc[split_idx]['date']

# Test = vraies valeurs originales uniquement
df_test = df_ml.iloc[split_idx:].copy()
X_test  = df_test[FEATURES]
y_test  = df_test[TARGET]

# Train = copies augmentées des mois train uniquement
train_dates  = set(df_ml.iloc[:split_idx]['date'])
df_train_aug = df_ml_aug[df_ml_aug['date'].isin(train_dates)].copy()
X_train      = df_train_aug[FEATURES]
y_train      = df_train_aug[TARGET]

# Scaler
scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train (augmenté) : {len(X_train)} lignes ({df_ml.iloc[:split_idx]["date"].min().strftime("%b %Y")} → {df_ml.iloc[:split_idx]["date"].max().strftime("%b %Y")})')
print(f'Test  (original) : {len(X_test)} mois ({df_test["date"].min().strftime("%b %Y")} → {df_test["date"].max().strftime("%b %Y")})')

# Visualiser
fig, ax = plt.subplots()
ax.plot(df_ml.iloc[:split_idx]['date'], df_ml.iloc[:split_idx][TARGET],
        label='Train (original)', color='steelblue', marker='o')
ax.plot(df_test['date'], y_test, label='Test', color='orange', marker='o')
ax.axvline(split_date, color='red', linestyle='--', label=f'Split : {split_date.strftime("%b %Y")}')
ax.set_title('📊 Split Temporel Train / Test')
ax.set_ylabel('CA (DT)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.legend()
plt.tight_layout()
plt.show()

## 10. Comparaison des modèles

In [ ]:
def evaluate(name, model, X_tr, y_tr, X_te, y_te):
    model.fit(X_tr, y_tr)
    pred  = model.predict(X_te)
    mae   = mean_absolute_error(y_te, pred)
    rmse  = np.sqrt(mean_squared_error(y_te, pred))
    r2    = r2_score(y_te, pred)
    mape  = np.mean(np.abs((y_te - pred) / y_te)) * 100
    smape = np.mean(2 * np.abs(y_te - pred) /
                   (np.abs(y_te) + np.abs(pred))) * 100
    bias  = np.mean(pred - y_te)
    return {
        'name': name, 'model': model, 'pred': pred,
        'MAE': mae, 'RMSE': rmse, 'R²': r2,
        'MAPE': mape, 'SMAPE': smape, 'Biais': bias
    }

models_config = [
    ('LinearRegression', LinearRegression(),                                                       X_train_sc, X_test_sc),
    ('Ridge',            Ridge(alpha=10),                                                          X_train_sc, X_test_sc),
    ('DecisionTree',     DecisionTreeRegressor(max_depth=3, random_state=42),                     X_train,    X_test),
    ('RandomForest',     RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42),   X_train,    X_test),
    ('XGBoost',          XGBRegressor(n_estimators=300, max_depth=3, learning_rate=0.03, random_state=42), X_train, X_test),
]

results = []
for name, model, Xtr, Xte in models_config:
    r = evaluate(name, model, Xtr, y_train, Xte, y_test)
    results.append(r)

df_results = pd.DataFrame(results).set_index('name')

print('=' * 85)
print(f'{"Modèle":<20} {"MAE":>10} {"RMSE":>10} {"R²":>8} {"MAPE":>8} {"SMAPE":>8} {"Biais":>12}')
print('=' * 85)
for _, row in df_results.iterrows():
    print(f'{row.name:<20} {row["MAE"]:>10,.0f} {row["RMSE"]:>10,.0f} '
          f'{row["R²"]:>8.4f} {row["MAPE"]:>7.2f}% {row["SMAPE"]:>7.2f}% {row["Biais"]:>12,.0f}')
print('=' * 85)

best_name = df_results['MAPE'].idxmin()
print(f'\n🏆 Meilleur modèle : {best_name} (MAPE = {df_results.loc[best_name, "MAPE"]:.2f}%)')

## 11. Visualisation Réel vs Prédit — tous les modèles

In [ ]:
fig, ax = plt.subplots()
ax.plot(df_test['date'], y_test.values,
        label='CA Réel', color='black', marker='o', linewidth=2.5)

colors = ['steelblue', 'orange', 'green', 'red', 'purple']
for (_, row), color in zip(df_results.iterrows(), colors):
    ax.plot(df_test['date'], row['pred'],
            label=f"{row.name} (MAPE={row['MAPE']:.1f}%)",
            marker='s', linestyle='--', color=color, alpha=0.8)

ax.set_title('📈 CA Réel vs Prédit — Comparaison Modèles')
ax.set_ylabel('CA (DT)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.legend()
plt.tight_layout()
plt.show()

## 12. Détail mois par mois — meilleur modèle

In [ ]:
best = df_results.loc[best_name]

df_detail = df_test[['date']].copy()
df_detail['reel']   = y_test.values
df_detail['predit'] = best['pred']
df_detail['ecart']  = best['pred'] - y_test.values
df_detail['ecart%'] = (df_detail['ecart'] / df_detail['reel'] * 100).round(2)

print(f'=== Détail {best_name} ===\n')
print(df_detail.to_string(index=False))
print(f"\nBiais : {best['Biais']:,.2f} DT  ({'surestimation' if best['Biais'] > 0 else 'sous-estimation'})")

## 13. Réentraîner sur tout le dataset augmenté + Sauvegarde
### Forcer LinearRegression pour prédiction stable

In [ ]:
print(f'Meilleur modèle test    : {best_name} (MAPE={best["MAPE"]:.2f}%)')
print(f'LinearRegression test   : MAPE={df_results.loc["LinearRegression", "MAPE"]:.2f}%, Biais={df_results.loc["LinearRegression", "Biais"]:,.0f} DT')
print(f'\nForce LinearRegression pour la prédiction (plus stable, biais ~0)\n')

# Réentraîner LinearRegression sur TOUT le dataset augmenté
X_full_aug = df_ml_aug[FEATURES]
y_full_aug = df_ml_aug[TARGET]

scaler_final = StandardScaler()
X_full_sc    = scaler_final.fit_transform(X_full_aug)
model_final  = LinearRegression()
model_final.fit(X_full_sc, y_full_aug)

# Sauvegarder
joblib.dump(model_final,  '/app/notebooks/best_model.pkl')
joblib.dump(scaler_final, '/app/notebooks/scaler.pkl')

with open('/app/notebooks/best_model_name.json', 'w') as f:
    json.dump({
        'name': 'LinearRegression',
        'mape': float(df_results.loc['LinearRegression', 'MAPE']),
        'trained_on': len(X_full_aug)
    }, f)

with open('/app/notebooks/features_corr.json', 'w') as f:
    json.dump(list(FEATURES), f)

print(f'✅ Modèle final sauvegardé : LinearRegression')
print(f'   Entraîné sur          : {len(X_full_aug)} lignes (augmenté)')
print(f'   MAPE test             : {df_results.loc["LinearRegression", "MAPE"]:.2f}%')
print(f'   Biais test            : {df_results.loc["LinearRegression", "Biais"]:,.0f} DT')

## 14. Prédiction — 12 mois futurs (Jan 2026 → Déc 2026)
### Contemporaines : moyenne du même mois sur les années passées

In [ ]:
import plotly.graph_objects as go

df_future = df_ml.copy()  # dataset original pour les lags réels
last_date = df_future['date'].max()  # Déc 2025
all_sales = list(df_future['total_sales'])

print(f'Modèle        : LinearRegression (réentraîné sur {len(X_full_aug)} lignes augmentées)')
print(f'Départ        : {last_date.strftime("%b %Y")}')
print(f'Objectif      : Jan 2026 → Déc 2026')
print(f'Contemporaines: {CONTEMPORAINES}\n')

predictions = []

for i in range(1, 13):
    next_date  = last_date + pd.DateOffset(months=i)
    next_month = next_date.month
    next_year  = next_date.year

    row = {}

    # ── Features temporelles
    if 'year'        in FEATURES: row['year']        = next_year
    if 'month'       in FEATURES: row['month']       = next_month
    if 'quarter'     in FEATURES: row['quarter']     = (next_month - 1) // 3 + 1
    if 'semester'    in FEATURES: row['semester']    = 1 if next_month <= 6 else 2
    if 'is_summer'   in FEATURES: row['is_summer']   = 1 if next_month in [7, 8] else 0
    if 'is_end_year' in FEATURES: row['is_end_year'] = 1 if next_month in [11, 12] else 0
    if 'is_january'  in FEATURES: row['is_january']  = 1 if next_month == 1 else 0

    # ── Lag features (depuis all_sales qui grandit)
    if 'lag_1'  in FEATURES: row['lag_1']  = all_sales[-1]
    if 'lag_12' in FEATURES:
        row['lag_12'] = all_sales[-12] if len(all_sales) >= 12 else np.mean(all_sales)

    # ── Rolling
    if 'rolling_mean_3' in FEATURES:
        row['rolling_mean_3'] = np.mean(all_sales[-3:])

    # ── Contemporaines → moyenne du même mois sur années passées
    # Ex: Juillet 2026 → moyenne Juillet 2022+2023+2024+2025
    for col in CONTEMPORAINES:
        same_month = df_future[df_future['month'] == next_month][col]
        row[col] = same_month.mean() if len(same_month) > 0 else df_future[col].mean()

    # ── Prédiction
    X_row = pd.DataFrame([row])[FEATURES]
    pred  = model_final.predict(scaler_final.transform(X_row))[0]

    all_sales.append(pred)
    predictions.append({
        'date':            next_date,
        'year':            next_year,
        'month':           next_month,
        'predicted_sales': pred,
        'is_prediction':   1
    })
    print(f'  {next_date.strftime("%b %Y")} → {pred:>15,.2f} DT')

df_pred = pd.DataFrame(predictions)
print(f'\n✅ {len(df_pred)} mois prédits')
print(f'CA prévisionnel 2026 : {df_pred["predicted_sales"].sum():,.2f} DT')

## 15. Visualisation — CA Réel + Prédit sur un seul graphe

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_ml['date'], y=df_ml['total_sales'],
    name='CA Réel', mode='lines+markers',
    line=dict(color='steelblue', width=2), marker=dict(size=5)
))

fig.add_trace(go.Scatter(
    x=df_pred['date'], y=df_pred['predicted_sales'],
    name='CA Prédit (LinearRegression)',
    mode='lines+markers',
    line=dict(color='orange', width=2, dash='dash'),
    marker=dict(size=6, symbol='square')
))

fig.add_vline(
    x=last_date.timestamp() * 1000,
    line_dash='dot', line_color='red',
    annotation_text=f'{last_date.strftime("%b %Y")} — Fin données réelles'
)

fig.update_layout(
    title='📈 CA Réel (Jan 2022 → Déc 2025) + Prédit (Jan 2026 → Déc 2026) — LinearRegression',
    xaxis_title='Mois', yaxis_title='CA (DT)',
    hovermode='x unified', height=500
)
fig.show()

## 16. CA annuel prévu vs réel

In [ ]:
ca_prevu   = df_pred['predicted_sales'].sum()
ca_2025    = df_ml[df_ml['date'].dt.year == 2025]['total_sales'].sum()
ca_2024    = df_ml[df_ml['date'].dt.year == 2024]['total_sales'].sum()
croissance = (ca_prevu - ca_2025) / ca_2025 * 100

print('=' * 55)
print('📋 PRÉVISION CA ANNUEL')
print('=' * 55)
print(f'CA réel 2024       : {ca_2024:>15,.2f} DT')
print(f'CA réel 2025       : {ca_2025:>15,.2f} DT')
print(f'CA prévu 2026      : {ca_prevu:>15,.2f} DT')
print(f'Croissance prévue  : {croissance:>15.2f} %')
print(f'Modèle             : LinearRegression')
print(f'MAPE test          : {df_results.loc["LinearRegression", "MAPE"]:.2f}%')
print('=' * 55)
print('\nDétail mensuel :')
print(df_pred[['date', 'predicted_sales']].to_string(index=False))

# Graphe annuel
years_data = pd.DataFrame([
    {'Année': '2024 (réel)',  'CA': ca_2024},
    {'Année': '2025 (réel)',  'CA': ca_2025},
    {'Année': '2026 (prédit)','CA': ca_prevu},
])
fig, ax = plt.subplots(figsize=(8, 5))
colors  = ['steelblue', 'steelblue', 'orange']
ax.bar(years_data['Année'], years_data['CA'], color=colors)
for i, (_, r) in enumerate(years_data.iterrows()):
    ax.text(i, r['CA'] + years_data['CA'].max() * 0.01,
            f"{r['CA']:,.0f} DT", ha='center', fontsize=9)
ax.set_title('📊 CA Annuel Réel vs Prévu')
ax.set_ylabel('CA (DT)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.show()

## 17. Écriture dans ClickHouse Gold

In [ ]:
client2 = clickhouse_connect.get_client(
    host='clickhouse', port=8123,
    user='default', password='clickhouse123',
    database='gold'
)

client2.command("""
    CREATE TABLE IF NOT EXISTS gold.ml_predictions (
        date             Date,
        year             Int32,
        month            Int32,
        predicted_sales  Float64,
        is_prediction    Int8
    )
    ENGINE = MergeTree()
    ORDER BY date
""")

client2.command('TRUNCATE TABLE gold.ml_predictions')
client2.insert_df('gold.ml_predictions',
    df_pred[['date', 'year', 'month', 'predicted_sales', 'is_prediction']])

check = client2.query_df('SELECT * FROM gold.ml_predictions ORDER BY date')
print(f'✅ {len(check)} prédictions écrites dans ClickHouse Gold')
print('   Table : gold.ml_predictions\n')
print(check.to_string(index=False))